# Week 2: Performance Evaluation & Embedding Comparison

This notebook implements the similarity pipeline and evaluates different embedding methods for the academic paper recommendation system.

**Tasks:**
1. Cross-Corpus Similarity (External ↔ BOUN)
2. Embedding Method Comparison (TF-IDF, Sentence Embeddings, Gemini API)
3. Intra-Paper Similarity (Same-Author & Cited Paper sanity checks)

In [40]:
import pandas as pd
import numpy as np
import json
import ast
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

pd.set_option('display.max_colwidth', 80)

## Load & Prepare Data

In [41]:

boun_data = pd.read_csv("data/cleaned/boun.csv")
priority_followed_data = pd.read_csv("data/cleaned/priority_followed.csv")

def safe_parse(x):
    """Safely parse stringified JSON/lists using ast.literal_eval with fallback."""
    if pd.isna(x):
        return None
    
    if isinstance(x, str):
        x = x.strip()
        if x.startswith(('[{', '[', '{', '(')):  # Detect literal structures
            try:
                return ast.literal_eval(x)
            except (ValueError, SyntaxError):
                # Fallback to string or attempt json.loads for valid JSON
                try:
                    return json.loads(x)
                except:
                    return x
    return x
    
print("Parsing BOUN data...")
for col in boun_data.columns:
    boun_data[col] = boun_data[col].map(safe_parse)

print("Parsing priority_followed data...")
for col in priority_followed_data.columns:
    priority_followed_data[col] = priority_followed_data[col].map(safe_parse)


Parsing BOUN data...
Parsing priority_followed data...


In [42]:
print(f"BOUN papers: {len(boun_data)}")
print(f"Followed institution papers: {len(priority_followed_data)}")

# Filter to papers with abstracts
boun_df = boun_data[boun_data['abstract'].notna() & (boun_data['abstract'].str.len() > 20)].reset_index(drop=True)
followed_df = priority_followed_data[priority_followed_data['abstract'].notna() & (priority_followed_data['abstract'].str.len() > 20)].reset_index(drop=True)

print(f"\nAfter filtering (with abstracts):")
print(f"BOUN papers: {len(boun_df)}")
print(f"Followed institution papers: {len(followed_df)}")

BOUN papers: 4608
Followed institution papers: 9946

After filtering (with abstracts):
BOUN papers: 3000
Followed institution papers: 5851


In [ ]:
def extract_concept_names(val):
    """Extract concept display names from the concepts column."""
    if isinstance(val, list):
        return ' '.join(c.get('display_name', '') for c in val if isinstance(c, dict))
    return ''

def build_text(row, fields=('abstract', 'title', 'concepts')):
    """Build text representation from selected fields.
    
    Parameters
    ----------
    fields : tuple of str
        Any combination of 'abstract', 'title', 'concepts'. Default: all three.
    """
    parts = []
    if 'title' in fields and isinstance(row.get('title'), str) and row['title']:
        parts.append(row['title'])
    if 'abstract' in fields and isinstance(row.get('abstract'), str) and row['abstract']:
        parts.append(row['abstract'])
    if 'concepts' in fields:
        concept_text = extract_concept_names(row.get('concepts', ''))
        if concept_text.strip():
            parts.append(concept_text)
    return ' '.join(parts)

# Pre-build combined text for default fields
boun_df['text'] = boun_df.apply(lambda r: build_text(r), axis=1)
followed_df['text'] = followed_df.apply(lambda r: build_text(r), axis=1)

Sample combined text:


0       Investigation of Social and Cultural Informal Learning Opportunities for Mid...
1       Rethinking ESG‐Financial Performance Nexus Beyond Linearity: A Configuration...
2       Massive runaway star HD 254577: The pre-supernova binary companion to the pr...
3       Evaluating Batch Correction Methods for Large-Scale Mass Spectrometry Imagin...
4       Machine learning for risk profiling: An analysis of pension fund participant...
                                             ...                                       
2995    Small-sized tourism projects in rural areas: the compounding effects on soci...
2996    Multifunctional and Transformable ‘Clickable’ Hydrogel Coatings on Titanium ...
2997    A Novel Bias-Eliminated Subspace Identification Approach for Closed-Loop Sys...
2998    Genetic Algorithm for the Mutual Information-Based Feature Selection in Univ...
2999    Complete Genome Sequence of <i>Pseudomonas</i> sp. Strain BIOMIG1 <sup>BAC</...
Name: text, Length: 3000, dtype:

In [48]:
print("Sample combined text:")
boun_df['text'][0]

Sample combined text:


'Investigation of Social and Cultural Informal Learning Opportunities for Middle and High School Students This study aims to investigate the social and cultural informal learning opportunities of middle and high school students based on gender, grade level, and place of residence. The sample consists of 1,310 students from a province located in eastern Türkiye. Data were collected through three items concerning personal information and two special multiple-response items in the Informal Learning Opportunities Test (ILOT). The data were analyzed using multiple response analysis and the chi-square test of independence. The results indicated that gender did not significantly influence students’ camp preferences; however, it had a significant effect on their activity preferences. Both boy and girl students exhibited low participation in science camps, yet girl students tended to prefer social, cultural, and nature-based activities, while boy students favored physical or technical activitie

## Similarity Methods

Each method implements the same interface: given query texts and corpus texts, return a similarity matrix.

### Method 1: TF-IDF + Cosine Similarity

In [44]:
def tfidf_similarity(query_texts, corpus_texts):
    """Compute TF-IDF cosine similarity between query and corpus texts.
    
    Returns: np.ndarray of shape (len(query_texts), len(corpus_texts))
    """
    vectorizer = TfidfVectorizer(max_features=50000, stop_words='english')
    all_texts = list(corpus_texts) + list(query_texts)
    tfidf_matrix = vectorizer.fit_transform(all_texts)
    
    corpus_vectors = tfidf_matrix[:len(corpus_texts)]
    query_vectors = tfidf_matrix[len(corpus_texts):]
    
    return cosine_similarity(query_vectors, corpus_vectors)

### Method 2: Sentence Embeddings (all-MiniLM-L6-v2 & SPECTER2)

In [45]:
def sentence_embedding_similarity(query_texts, corpus_texts, model_name='all-MiniLM-L6-v2', batch_size=256):
    """Compute cosine similarity using sentence transformer embeddings.
    
    Supported models: 'all-MiniLM-L6-v2', 'allenai/specter2_base'
    Returns: np.ndarray of shape (len(query_texts), len(corpus_texts))
    """
    model = SentenceTransformer(model_name)
    
    print(f"  Encoding {len(corpus_texts)} corpus texts...")
    corpus_embeddings = model.encode(list(corpus_texts), batch_size=batch_size, show_progress_bar=True)
    
    print(f"  Encoding {len(query_texts)} query texts...")
    query_embeddings = model.encode(list(query_texts), batch_size=batch_size, show_progress_bar=True)
    
    return cosine_similarity(query_embeddings, corpus_embeddings)

### Method 3: Gemini API Similarity Scoring

Uses the Gemini API to directly score semantic similarity between paper pairs. Due to API rate limits, this is applied only to the top candidates pre-filtered by a fast method.

In [49]:
import google.generativeai as genai
import time
import os

# Configure Gemini API - set your API key here or via environment variable
# genai.configure(api_key="YOUR_API_KEY")
genai.configure(api_key=os.environ.get("GEMINI_API_KEY", ""))

def gemini_score_pair(query_text, candidate_text, model_name="gemini-2.0-flash"):
    """Use Gemini to score semantic similarity between two papers (0-100)."""
    prompt = f"""Rate the semantic similarity between these two academic papers on a scale of 0-100, 
where 0 means completely unrelated and 100 means nearly identical topics.

Paper A:
{query_text[:1500]}

Paper B:
{candidate_text[:1500]}

Respond with ONLY a single integer score (0-100), nothing else."""
    
    model = genai.GenerativeModel(model_name)
    try:
        response = model.generate_content(prompt)
        score = int(response.text.strip())
        return min(max(score, 0), 100) / 100.0
    except Exception as e:
        print(f"  Gemini error: {e}")
        return None

def gemini_rerank(query_text, candidate_texts, candidate_indices, top_k=3):
    """Re-rank top candidates using Gemini scoring.
    
    Returns list of (index, score) tuples sorted by Gemini score descending.
    """
    scored = []
    for idx, cand_text in zip(candidate_indices, candidate_texts):
        score = gemini_score_pair(query_text, cand_text)
        if score is not None:
            scored.append((idx, score))
        time.sleep(0.5)  # Rate limit
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

print("Gemini API configured.")

Gemini API configured.


### Retrieval Helper

Generic function to retrieve top-K similar papers using any method.

In [50]:
def retrieve_top_k(query_texts, corpus_texts, method, top_k=3, **kwargs):
    """Retrieve top-K similar corpus papers for each query.
    
    Parameters
    ----------
    method : str — 'tfidf', 'minilm', 'specter2', or 'gemini'
    Returns: list of list of (corpus_index, score)
    """
    if method == 'tfidf':
        sim_matrix = tfidf_similarity(query_texts, corpus_texts)
    elif method == 'minilm':
        sim_matrix = sentence_embedding_similarity(query_texts, corpus_texts, model_name='all-MiniLM-L6-v2')
    elif method == 'specter2':
        sim_matrix = sentence_embedding_similarity(query_texts, corpus_texts, model_name='allenai/specter2_base')
    elif method == 'gemini':
        pre_k = kwargs.get('pre_filter_k', 10)
        sim_matrix = tfidf_similarity(query_texts, corpus_texts)
        results = []
        for i in range(len(query_texts)):
            top_indices = np.argsort(sim_matrix[i])[::-1][:pre_k]
            cand_texts = [corpus_texts[j] for j in top_indices]
            reranked = gemini_rerank(query_texts[i], cand_texts, top_indices, top_k=top_k)
            results.append(reranked)
        return results
    else:
        raise ValueError(f"Unknown method: {method}")
    
    results = []
    for i in range(sim_matrix.shape[0]):
        top_indices = np.argsort(sim_matrix[i])[::-1][:top_k]
        results.append([(idx, sim_matrix[i, idx]) for idx in top_indices])
    return results

print("retrieve_top_k() ready.")

retrieve_top_k() ready.


## Task 1 — Cross-Corpus Similarity

**1a.** 50 followed-institution papers → top-3 similar BOUN papers  
**1b.** 50 BOUN papers → top-3 similar followed-institution papers

In [ ]:
np.random.seed(42)
sample_followed = followed_df.sample(50).reset_index(drop=True)
sample_boun = boun_df.sample(50).reset_index(drop=True)

METHODS = ['tfidf', 'minilm', 'specter2']

# --- Task 1a: External → BOUN ---
print("=" * 80)
print("TASK 1a: External → BOUN (50 followed-institution queries → BOUN corpus)")
print("=" * 80)

results_1a = {}
for method in METHODS:
    print(f"\n▶ Running {method}...")
    results_1a[method] = retrieve_top_k(
        list(sample_followed['text']),
        list(boun_df['text']),
        method=method,
        top_k=3
    )

# Display side-by-side comparison for first 5 queries
for i in range(5):
    query_title = str(sample_followed.iloc[i]['title'])[:80]
    print(f"\n{'─'*80}")
    print(f"Query {i+1}: {query_title}")
    for method in METHODS:
        hits = results_1a[method][i]
        print(f"  [{method:>10}]")
        for rank, (idx, score) in enumerate(hits, 1):
            match_title = str(boun_df.iloc[idx]['title'])[:60]
            print(f"    #{rank} ({score:.3f}) {match_title}")

TASK 1a: External → BOUN (50 followed-institution queries → BOUN corpus)

▶ Running tfidf...

▶ Running minilm...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5211.51it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Encoding 3000 corpus texts...


Batches: 100%|██████████| 12/12 [00:19<00:00,  1.61s/it]


  Encoding 50 query texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.55it/s]



▶ Running specter2...


No sentence-transformers model found with name allenai/specter2_base. Creating a new one with mean pooling.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17232.36it/s]
BertModel LOAD REPORT from: allenai/specter2_base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Encoding 3000 corpus texts...


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

In [ ]:
# --- Task 1b: BOUN → External ---
print("=" * 80)
print("TASK 1b: BOUN → External (50 BOUN queries → followed-institution corpus)")
print("=" * 80)

results_1b = {}
for method in METHODS:
    print(f"\n▶ Running {method}...")
    results_1b[method] = retrieve_top_k(
        list(sample_boun['text']),
        list(followed_df['text']),
        method=method,
        top_k=3
    )

# Display side-by-side comparison for first 5 queries
for i in range(5):
    query_title = str(sample_boun.iloc[i]['title'])[:80]
    print(f"\n{'─'*80}")
    print(f"Query {i+1}: {query_title}")
    for method in METHODS:
        hits = results_1b[method][i]
        print(f"  [{method:>10}]")
        for rank, (idx, score) in enumerate(hits, 1):
            match_title = str(followed_df.iloc[idx]['title'])[:60]
            print(f"    #{rank} ({score:.3f}) {match_title}")

## Task 2 — Embedding Method Comparison

Evaluate all methods against the **benchmark pairs** from Week 1.  
**Metrics:** Hit Rate@K (fraction of queries where ≥1 positive is in top-K), Mean Reciprocal Rank (MRR).

In [ ]:
# Load benchmark pairs
with open('eval_dataset/benchmark_pairs_with_related_works.json') as f:
    benchmark_pairs = json.load(f)

print(f"Loaded {len(benchmark_pairs)} benchmark query-positive pairs")

# Build lookup: BOUN paper id → index in boun_df
boun_id_to_idx = {str(row['id']): i for i, row in boun_df.iterrows()}

# Build query texts and collect expected positive BOUN paper ids
bench_queries = []
bench_positive_ids = []

for pair in benchmark_pairs:
    query = pair['query']
    q_text = f"{query.get('title', '')} {query.get('abstract', '')}"
    bench_queries.append(q_text)
    bench_positive_ids.append([p['id'] for p in pair['positives']])

print(f"Benchmark queries: {len(bench_queries)}")
print(f"Avg positives per query: {np.mean([len(p) for p in bench_positive_ids]):.1f}")